# Build llama.cpp (CUDA)

Run this notebook **in parallel** with the main training notebook.
It clones and compiles llama.cpp with CUDA support, then copies the quantizer binary to /workspace.

Set **CUDA_ARCH** to match your GPU:
- 89 = RTX 4080S / 4090 (Ada Lovelace)
- 86 = RTX 3090 / 3080 (Ampere)
- 120 = RTX 5090 (Blackwell)

In [ ]:
import subprocess, shutil, sys, os
from pathlib import Path

# ── Set this to match your GPU ──────────────────────────────────────────────
# 89 = RTX 4080S / 4090 (Ada Lovelace)
# 86 = RTX 3090 / 3080 (Ampere)
# 120 = RTX 5090 (Blackwell)
CUDA_ARCH = 89

LLAMA_BUILD_DIR = Path("/workspace/llama.cpp")

if not LLAMA_BUILD_DIR.exists():
    print("Cloning llama.cpp ...")
    subprocess.run(["git", "clone", "https://github.com/ggerganov/llama.cpp", str(LLAMA_BUILD_DIR)], check=True)
else:
    print("llama.cpp already cloned, skipping.")

quantizer_bin = LLAMA_BUILD_DIR / "build" / "bin" / "llama-quantize"
if not quantizer_bin.exists():
    print(f"Building llama.cpp with CUDA arch {CUDA_ARCH} ...")
    subprocess.run(
        ["cmake", "-B", "build", "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}"],
        cwd=str(LLAMA_BUILD_DIR), check=True
    )
    subprocess.run(
        ["cmake", "--build", "build", "--config", "Release", f"-j{os.cpu_count()}"],
        cwd=str(LLAMA_BUILD_DIR), check=True
    )
    print("Build complete.")
else:
    print("llama.cpp already built, skipping.")

# Copy quantizer to /workspace so it can be executed (build dir may have noexec issues)
quantizer_dest = Path("/workspace/llama-quantize")
shutil.copy2(str(quantizer_bin), str(quantizer_dest))
quantizer_dest.chmod(0o755)
print(f"Quantizer ready: {quantizer_dest}")